<a href="https://colab.research.google.com/github/Muhammad-Faiz-Firmansyah/2026_Pemrograman-Berorientasi-Objek/blob/main/jobsheet11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jobsheet 11 - Integrasi OOP dalam Aplikasi Pengeluaran Sederhana
Notebook ini membuat aplikasi pencatat pengeluaran harian dengan OOP, SQLite, Pandas, dan Streamlit.

## Membuat `konfigurasi.py`

In [ ]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ['Makanan', 'Transportasi', 'Hiburan', 'Tagihan', 'Belanja', 'Kesehatan', 'Pendidikan', 'Lainnya']
KATEGORI_DEFAULT = 'Lainnya'

Overwriting konfigurasi.py


## Membuat `database.py`

In [ ]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection():
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f'ERROR koneksi DB gagal: {e}')
        return None

def execute_query(query, params=None):
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        cursor.execute(query, params or ())
        conn.commit()
        return cursor.lastrowid
    except sqlite3.Error as e:
        print(f'ERROR query gagal: {e}')
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query, params=None, fetch_all=True):
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        cursor.execute(query, params or ())
        return cursor.fetchall() if fetch_all else cursor.fetchone()
    except sqlite3.Error as e:
        print(f'ERROR fetch gagal: {e}')
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query, params=None):
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()
    try:
        return pd.read_sql_query(query, conn, params=params)
    except Exception as e:
        print(f'ERROR baca DataFrame gagal: {e}')
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    query = '''CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL CHECK(jumlah > 0),
        kategori TEXT,
        tanggal DATE NOT NULL
    );'''
    return execute_query(query) is not None

Overwriting database.py


## Membuat `model.py`

In [ ]:
%%writefile model.py
import datetime
from konfigurasi import KATEGORI_DEFAULT

class Transaksi:
    def __init__(self, deskripsi, jumlah, kategori, tanggal, id_transaksi=None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi).strip() if deskripsi else 'Tanpa Deskripsi'
        try:
            jumlah = float(jumlah)
            if jumlah <= 0:
                raise ValueError('Jumlah harus positif')
            self.jumlah = jumlah
        except (ValueError, TypeError):
            raise ValueError('Jumlah transaksi harus angka positif')
        self.kategori = str(kategori).strip() if kategori else KATEGORI_DEFAULT
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            self.tanggal = datetime.datetime.strptime(tanggal, '%Y-%m-%d').date()
        else:
            self.tanggal = datetime.date.today()

    def __repr__(self):
        return f'Transaksi(ID:{self.id}, Tgl:{self.tanggal}, Jml:{self.jumlah:.0f}, Kat:\"{self.kategori}\", Desc:\"{self.deskripsi}\")'

    def to_dict(self):
        return {
            'deskripsi': self.deskripsi,
            'jumlah': self.jumlah,
            'kategori': self.kategori,
            'tanggal': self.tanggal.strftime('%Y-%m-%d')
        }

Overwriting model.py


## Membuat `manajer_anggaran.py`

In [ ]:
%%writefile manajer_anggaran.py
import pandas as pd
from database import execute_query, fetch_query, get_dataframe, setup_database_initial
from model import Transaksi

class AnggaranHarian:
    def __init__(self):
        setup_database_initial()

    def tambah_transaksi(self, transaksi: Transaksi):
        query = 'INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)'
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime('%Y-%m-%d'))
        return execute_query(query, params)

    def hapus_transaksi(self, id_transaksi: int) -> bool:
        query = 'DELETE FROM transaksi WHERE id = ?'
        hasil = execute_query(query, (id_transaksi,))
        return hasil is not None

    def get_semua_transaksi(self):
        return get_dataframe('SELECT id, tanggal, deskripsi, kategori, jumlah FROM transaksi ORDER BY tanggal DESC, id DESC')

    def hitung_total_pengeluaran(self, filter_tanggal=None):
        if filter_tanggal:
            row = fetch_query('SELECT SUM(jumlah) AS total FROM transaksi WHERE tanggal = ?', (str(filter_tanggal),), False)
        else:
            row = fetch_query('SELECT SUM(jumlah) AS total FROM transaksi', fetch_all=False)
        return float(row['total'] or 0) if row else 0.0

    def get_pengeluaran_per_kategori(self, filter_tanggal=None):
        if filter_tanggal:
            return get_dataframe('SELECT kategori, SUM(jumlah) AS total FROM transaksi WHERE tanggal = ? GROUP BY kategori ORDER BY total DESC', (str(filter_tanggal),))
        return get_dataframe('SELECT kategori, SUM(jumlah) AS total FROM transaksi GROUP BY kategori ORDER BY total DESC')


Overwriting manajer_anggaran.py


## Membuat `setup_db_pengeluaran.py`

In [ ]:
%%writefile setup_db_pengeluaran.py
from database import setup_database_initial
from konfigurasi import DB_PATH

if __name__ == '__main__':
    print('--- Setup Database Pengeluaran ---')
    if setup_database_initial():
        print(f'Database siap: {DB_PATH}')
    else:
        print('Setup database gagal')

Overwriting setup_db_pengeluaran.py


## Membuat `streamlit_app.py`

In [ ]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import altair as alt
from konfigurasi import KATEGORI_PENGELUARAN, KATEGORI_DEFAULT
from model import Transaksi
from manajer_anggaran import AnggaranHarian

st.set_page_config(page_title='Pengeluaran Harian', page_icon='💰', layout='wide')
st.markdown('''
<style>
.main-title {font-size: 2.4rem; font-weight: 800; margin-bottom: .2rem;}
.subtitle {color: #9ca3af; margin-bottom: 1.2rem;}
</style>
''', unsafe_allow_html=True)

st.markdown('<div class="main-title">💰 Pengeluaran Harian</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Catat transaksi, pantau total, lihat grafik, dan hapus transaksi yang salah input.</div>', unsafe_allow_html=True)

anggaran = AnggaranHarian()

with st.sidebar:
    st.header('➕ Tambah Transaksi')
    with st.form('form_transaksi', clear_on_submit=True):
        deskripsi = st.text_input('Deskripsi', placeholder='Contoh: Makan siang')
        jumlah = st.number_input('Jumlah (Rp)', min_value=1.0, step=1000.0, format='%.0f')
        kategori = st.selectbox('Kategori', KATEGORI_PENGELUARAN, index=KATEGORI_PENGELUARAN.index(KATEGORI_DEFAULT))
        tanggal = st.date_input('Tanggal')
        submitted = st.form_submit_button('💾 Simpan', use_container_width=True)
        if submitted:
            try:
                trx = Transaksi(deskripsi, jumlah, kategori, tanggal)
                anggaran.tambah_transaksi(trx)
                st.cache_data.clear()
                st.success('Transaksi berhasil disimpan')
                st.rerun()
            except Exception as e:
                st.error(f'Gagal menyimpan: {e}')

filter_tanggal = st.date_input('Filter tanggal (opsional)', value=None)
df = anggaran.get_semua_transaksi()
kat_df = anggaran.get_pengeluaran_per_kategori(filter_tanggal if filter_tanggal else None)

col1, col2, col3 = st.columns(3)
col1.metric('Total Semua Pengeluaran', f'Rp {anggaran.hitung_total_pengeluaran():,.0f}'.replace(',', '.'))
col2.metric('Total Tanggal Terpilih', f'Rp {anggaran.hitung_total_pengeluaran(filter_tanggal):,.0f}'.replace(',', '.') if filter_tanggal else 'Pilih tanggal')
col3.metric('Jumlah Transaksi', len(df))

st.divider()
tab_ringkasan, tab_riwayat = st.tabs(['📊 Ringkasan', '📋 Riwayat Lengkap'])

with tab_ringkasan:
    st.subheader('Pengeluaran per Kategori')
    if not kat_df.empty:
        chart_df = kat_df.sort_values('total', ascending=True).copy()
        chart_df['label'] = chart_df['total'].apply(lambda x: f"Rp {x:,.0f}".replace(',', '.'))
        chart = (
            alt.Chart(chart_df)
            .mark_bar(cornerRadiusTopRight=8, cornerRadiusBottomRight=8)
            .encode(
                y=alt.Y('kategori:N', sort='-x', title=None),
                x=alt.X('total:Q', title='Total Pengeluaran (Rp)'),
                color=alt.Color('kategori:N', legend=None, scale=alt.Scale(scheme='set2')),
                tooltip=[alt.Tooltip('kategori:N', title='Kategori'), alt.Tooltip('label:N', title='Total')]
            )
            .properties(height=320)
        )
        text = alt.Chart(chart_df).mark_text(align='left', dx=6, color='white').encode(
            y=alt.Y('kategori:N', sort='-x'), x='total:Q', text='label:N'
        )
        st.altair_chart(chart + text, use_container_width=True)
    else:
        st.info('Belum ada data untuk grafik')

with tab_riwayat:
    st.subheader('Riwayat Lengkap Transaksi')
    if not df.empty:
        tampil = df.copy()
        tampil['jumlah'] = tampil['jumlah'].apply(lambda x: f"Rp {x:,.0f}".replace(',', '.'))
        st.dataframe(tampil, use_container_width=True, hide_index=True)

        st.markdown('### 🗑️ Hapus Transaksi')
        id_hapus = st.number_input('ID Transaksi Hapus:', min_value=1, step=1)
        if st.button('Hapus Transaksi Terpilih', type='secondary'):
            st.session_state['konfirmasi_hapus_id'] = int(id_hapus)

        if 'konfirmasi_hapus_id' in st.session_state:
            id_konfirmasi = st.session_state['konfirmasi_hapus_id']
            st.warning(f'Yakin ingin menghapus transaksi dengan ID {id_konfirmasi}?')
            col_ok, col_batal = st.columns(2)
            with col_ok:
                if st.button('✅ Konfirmasi Hapus', type='primary'):
                    berhasil = anggaran.hapus_transaksi(id_konfirmasi)
                    if berhasil:
                        st.success(f'Transaksi ID {id_konfirmasi} berhasil dihapus')
                        st.session_state.pop('konfirmasi_hapus_id', None)
                        st.cache_data.clear()
                        st.rerun()
                    else:
                        st.error('Transaksi gagal dihapus')
            with col_batal:
                if st.button('Batal'):
                    st.session_state.pop('konfirmasi_hapus_id', None)
                    st.rerun()
    else:
        st.info('Belum ada data transaksi')


Overwriting streamlit_app.py


## Setup database dan uji cepat

In [ ]:
!python setup_db_pengeluaran.py
from manajer_anggaran import AnggaranHarian
from model import Transaksi
import datetime
manager = AnggaranHarian()
manager.tambah_transaksi(Transaksi("Makan siang", 25000, "Makanan", datetime.date.today()))
manager.get_semua_transaksi()


--- Setup Database Pengeluaran ---
Database siap: /content/pengeluaran_harian.db


,id,tanggal,deskripsi,kategori,jumlah
0,6,2026-06-11,Makan siang,Makanan,25000.0
1,5,2026-06-11,Tanpa Deskripsi,Lainnya,1.0
2,4,2026-06-11,Tanpa Deskripsi,Makanan,100000.0
3,3,2026-06-11,Makan siang,Makanan,25000.0
4,2,2026-06-11,Makan siang,Makanan,25000.0
5,1,2026-06-11,Makan siang,Makanan,25000.0


## Menjalankan Streamlit di Google Colab
Di Colab, Streamlit perlu tunnel agar punya link public. Jalankan 2 sel di bawah ini. Link public akan muncul dari `npx localtunnel`.

In [ ]:
!pip -q install streamlit pandas altair
!npm install -g localtunnel


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

## Menjalankan Streamlit di Google Colab (Stable Version)
Kita menggunakan Cloudflared untuk tunnel karena lebih stabil dalam memuat aset UI (menghindari error TypeError/Failed to fetch).

In [ ]:
!pip -q install streamlit pandas altair
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-06-11 02:25:35--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.6.0/cloudflared-linux-amd64.deb [following]
--2026-06-11 02:25:35--  https://github.com/cloudflare/cloudflared/releases/download/2026.6.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/02570f6f-b171-47dc-9456-07ba41559e6b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-11T03%3A15%3A42Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [ ]:
import subprocess, time
streamlit_process = subprocess.Popen(["streamlit", "run", "streamlit_app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)
print("Streamlit berjalan di port 8501. Jalankan sel berikutnya untuk link public.")


Streamlit berjalan di port 8501. Jalankan sel berikutnya untuk link public.


In [ ]:
!cloudflared tunnel --url http://localhost:8501


2026-06-11T02:25:42Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-11T02:25:42Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-11T02:25:45Z INF +--------------------------------------------------------------------------------------------+
2026-06-11T02:25:45Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-11T02:25:45Z INF |  https://reflect-atmosphere-wales-locations.trycloudfl